# Enhanced Data Merging - Create Master Dataset

**Purpose:** Merge all datasets into one comprehensive feature set for electricity demand forecasting.

**Datasets to merge:**

1. **Base Dataset** (30 columns, hourly):
   - IESO demand + weather + engineered features
   - Source: `ieso_weather_engineered_features.csv`

2. **HOEP Prices** (hourly):
   - Hourly Ontario Energy Price ($/MWh)
   - Source: `hoep_prices_2013_2025.csv`

3. **NASA Solar Data** (daily):
   - GHI, DNI, DHI solar irradiance (kWh/m²/day)
   - Source: `nasa_solar_2013_2025.csv`
   - **Note:** Daily data will be applied to all hours in each day

4. **Ontario Holidays** (118 dates):
   - Statutory holiday flags
   - Source: `ontario_holidays_2013_2025.csv`
   - **Note:** Create binary indicator for each hour

**Expected final dataset:** ~38-40 columns, 109,056 hourly records

**Merge strategy:**
- HOEP: Direct hourly merge on DateTime
- Solar: Merge on Date (daily values applied to all hours)
- Holidays: Merge on Date, create binary flags

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

# Load base dataset (engineered features)
print("Loading base dataset with engineered features...")
df = pd.read_csv("../../02_Datasets/processed/ieso_weather_engineered_features.csv", 
                 parse_dates=['DateTime', 'Date'])

print(f"✓ Base dataset loaded")
print(f"  Shape: {df.shape}")
print(f"  Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"  Columns: {len(df.columns)}")

# Show first few rows
print(f"\nFirst 3 rows:")
df.head(3)

Loading base dataset with engineered features...
✓ Base dataset loaded
  Shape: (109056, 30)
  Date range: 2013-06-01 00:00:00 to 2025-11-09 00:00:00
  Columns: 30

First 3 rows:


,DateTime,Date,Hour,Year,Month,DayOfWeek,DayOfYear,IsWeekend,Market Demand,Ontario Demand,Temp (°C),Dew Point Temp (°C),Rel Hum (%),Wind Dir (10s deg),Wind Spd (km/h),Visibility (km),Stn Press (kPa),Demand_Lag_1h,Demand_Lag_24h,Demand_Lag_168h,Demand_RollingAvg_24h,Demand_RollingAvg_168h,HDD,CDD,Temp_x_Hour,Temp_Squared,Hour_Sin,Hour_Cos,Month_Sin,Month_Cos
0,2013-06-01 00:00:00,2013-06-01,1,2013,6,5,152,1,15908,13731,17.8,16.0,89.0,0.0,21.0,24.1,98.66,NaN,NaN,NaN,13731.000000,13731.000000,0.2,0.0,17.8,316.84,0.258819,0.965926,1.224647e-16,-1.0
1,2013-06-01 01:00:00,2013-06-01,2,2013,6,5,152,1,15126,13143,17.8,16.0,89.0,0.0,21.0,24.1,98.66,13731.0,NaN,NaN,13437.000000,13437.000000,0.2,0.0,35.6,316.84,0.500000,0.866025,1.224647e-16,-1.0
2,2013-06-01 02:00:00,2013-06-01,3,2013,6,5,152,1,14652,12778,17.8,16.0,89.0,0.0,21.0,24.1,98.66,13143.0,NaN,NaN,13217.333333,13217.333333,0.2,0.0,53.4,316.84,0.707107,0.707107,1.224647e-16,-1.0


In [2]:
print("Loading and merging HOEP prices...")

# Load HOEP prices
df_hoep = pd.read_csv("../../02_Datasets/processed/hoep_prices_2013_2025.csv",
                      parse_dates=['DateTime'])

print(f"✓ HOEP data loaded: {df_hoep.shape}")
print(f"  Date range: {df_hoep['DateTime'].min()} to {df_hoep['DateTime'].max()}")

# Merge HOEP with base dataset on DateTime
df = df.merge(df_hoep, on='DateTime', how='left')

print(f"\n✓ HOEP prices merged")
print(f"  New shape: {df.shape}")
print(f"  Missing HOEP values: {df['HOEP'].isna().sum()} (expected for dates after April 2025)")

# Show example
print(f"\nExample with HOEP:")
print(df[['DateTime', 'Ontario Demand', 'HOEP']].head(5))

Loading and merging HOEP prices...
✓ HOEP data loaded: (104449, 2)
  Date range: 2013-06-01 00:00:00 to 2025-05-01 00:00:00

✓ HOEP prices merged
  New shape: (109056, 31)
  Missing HOEP values: 4613 (expected for dates after April 2025)

Example with HOEP:
             DateTime  Ontario Demand   HOEP
0 2013-06-01 00:00:00           13731  13.86
1 2013-06-01 01:00:00           13143  12.73
2 2013-06-01 02:00:00           12778  11.21
3 2013-06-01 03:00:00           12509   6.65
4 2013-06-01 04:00:00           12574  -3.62


In [3]:
print("Loading and merging NASA solar data...")

# Load NASA solar data
df_solar = pd.read_csv("../../02_Datasets/processed/nasa_solar_2013_2025.csv",
                       parse_dates=['Date'])

print(f"✓ Solar data loaded: {df_solar.shape}")
print(f"  Date range: {df_solar['Date'].min()} to {df_solar['Date'].max()}")

# Merge solar data on Date (daily values applied to all hours in that day)
df = df.merge(df_solar, on='Date', how='left')

print(f"\n✓ Solar data merged")
print(f"  New shape: {df.shape}")
print(f"  Missing GHI values: {df['GHI'].isna().sum()}")
print(f"  Missing DNI values: {df['DNI'].isna().sum()}")
print(f"  Missing DHI values: {df['DHI'].isna().sum()}")

# Show example
print(f"\nExample with solar data:")
print(df[['DateTime', 'Date', 'GHI', 'DNI', 'DHI']].head(10))

Loading and merging NASA solar data...
✓ Solar data loaded: (4552, 4)
  Date range: 2013-06-01 00:00:00 to 2025-11-16 00:00:00

✓ Solar data merged
  New shape: (109056, 34)
  Missing GHI values: 24
  Missing DNI values: 2425
  Missing DHI values: 2425

Example with solar data:
             DateTime       Date     GHI    DNI     DHI
0 2013-06-01 00:00:00 2013-06-01  4.4316  1.219  3.3902
1 2013-06-01 01:00:00 2013-06-01  4.4316  1.219  3.3902
2 2013-06-01 02:00:00 2013-06-01  4.4316  1.219  3.3902
3 2013-06-01 03:00:00 2013-06-01  4.4316  1.219  3.3902
4 2013-06-01 04:00:00 2013-06-01  4.4316  1.219  3.3902
5 2013-06-01 05:00:00 2013-06-01  4.4316  1.219  3.3902
6 2013-06-01 06:00:00 2013-06-01  4.4316  1.219  3.3902
7 2013-06-01 07:00:00 2013-06-01  4.4316  1.219  3.3902
8 2013-06-01 08:00:00 2013-06-01  4.4316  1.219  3.3902
9 2013-06-01 09:00:00 2013-06-01  4.4316  1.219  3.3902


In [4]:
print("Loading and merging holiday data...")

# Load holiday data
df_holidays = pd.read_csv("../../02_Datasets/processed/ontario_holidays_2013_2025.csv",
                          parse_dates=['Date'])

print(f"✓ Holiday data loaded: {df_holidays.shape}")
print(f"  Total holidays: {len(df_holidays)}")

# Create a binary holiday indicator
df_holidays['IsHoliday'] = 1

# Merge holidays on Date (left join to keep all hours)
df = df.merge(df_holidays[['Date', 'IsHoliday', 'Holiday_Name']], 
              on='Date', how='left')

# Fill NaN with 0 for non-holiday days
df['IsHoliday'] = df['IsHoliday'].fillna(0).astype(int)
df['Holiday_Name'] = df['Holiday_Name'].fillna('')

print(f"\n✓ Holiday data merged")
print(f"  New shape: {df.shape}")
print(f"  Holiday hours: {df['IsHoliday'].sum():,} ({df['IsHoliday'].sum() / len(df) * 100:.2f}%)")
print(f"  Non-holiday hours: {(df['IsHoliday'] == 0).sum():,}")

# Show examples of holidays
print(f"\nExample holiday hours:")
holiday_examples = df[df['IsHoliday'] == 1].head(10)
print(holiday_examples[['DateTime', 'Date', 'IsHoliday', 'Holiday_Name']])

Loading and merging holiday data...
✓ Holiday data loaded: (118, 2)
  Total holidays: 118

✓ Holiday data merged
  New shape: (109056, 36)
  Holiday hours: 2,784 (2.55%)
  Non-holiday hours: 106,272

Example holiday hours:
               DateTime       Date  IsHoliday Holiday_Name
720 2013-07-01 00:00:00 2013-07-01          1   Canada Day
721 2013-07-01 01:00:00 2013-07-01          1   Canada Day
722 2013-07-01 02:00:00 2013-07-01          1   Canada Day
723 2013-07-01 03:00:00 2013-07-01          1   Canada Day
724 2013-07-01 04:00:00 2013-07-01          1   Canada Day
725 2013-07-01 05:00:00 2013-07-01          1   Canada Day
726 2013-07-01 06:00:00 2013-07-01          1   Canada Day
727 2013-07-01 07:00:00 2013-07-01          1   Canada Day
728 2013-07-01 08:00:00 2013-07-01          1   Canada Day
729 2013-07-01 09:00:00 2013-07-01          1   Canada Day


In [5]:
# Final check on missing values
print("Final Dataset Summary")
print("="*70)

print(f"\nShape: {df.shape}")
print(f"Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"Total records: {len(df):,}")

print(f"\nMissing values by column:")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print(missing)
else:
    print("  No missing values!")

print(f"\nTotal missing values: {df.isna().sum().sum():,}")

# Save the final master dataset
output_path = "../../02_Datasets/processed/master_dataset_complete.csv"
df.to_csv(output_path, index=False)

print(f"\n{'='*70}")
print(f"✓ MASTER DATASET SAVED!")
print(f"{'='*70}")
print(f"\nFile: {output_path}")
print(f"Size: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nFinal feature count: {len(df.columns)} columns")
print(f"\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

Final Dataset Summary

Shape: (109056, 36)
Date range: 2013-06-01 00:00:00 to 2025-11-09 00:00:00
Total records: 109,056

Missing values by column:
HOEP               4613
DNI                2425
DHI                2425
Demand_Lag_168h     168
Demand_Lag_24h       24
GHI                  24
Demand_Lag_1h         1
dtype: int64

Total missing values: 9,680

✓ MASTER DATASET SAVED!

File: ../../02_Datasets/processed/master_dataset_complete.csv
Size: 33.84 MB

Final feature count: 36 columns

All columns:
   1. DateTime
   2. Date
   3. Hour
   4. Year
   5. Month
   6. DayOfWeek
   7. DayOfYear
   8. IsWeekend
   9. Market Demand
  10. Ontario Demand
  11. Temp (°C)
  12. Dew Point Temp (°C)
  13. Rel Hum (%)
  14. Wind Dir (10s deg)
  15. Wind Spd (km/h)
  16. Visibility (km)
  17. Stn Press (kPa)
  18. Demand_Lag_1h
  19. Demand_Lag_24h
  20. Demand_Lag_168h
  21. Demand_RollingAvg_24h
  22. Demand_RollingAvg_168h
  23. HDD
  24. CDD
  25. Temp_x_Hour
  26. Temp_Squared
  27. Hour_Sin
